# 🕌 Shariah Efficient Portfolio — IA Quantamentale

**Stack IA complète :** HMM Régimes • XGBoost + SHAP • LSTM Volatilité • FinBERT NLP • PPO Agent RL • Black-Litterman

**Backtest :** 20 ans walk-forward • EU + US • Fiscalité CTO France • Multi-broker

[![GitHub](https://img.shields.io/badge/GitHub-Source-black)](https://github.com/VOTRE_USERNAME/shariah-portfolio)

---
### ⚡ Instructions
1. **Runtime → Changer le type de runtime → GPU T4** (gratuit)
2. Exécuter les cellules dans l'ordre
3. Configurer vos paramètres dans la **Cellule 3**
4. Lancer le backtest complet


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 1 : Installation des dépendances (~3 minutes)
# ═══════════════════════════════════════════════════════════
import subprocess, sys

print('📦 Installation des packages...')
packages = [
    'yfinance>=0.2.36',
    'pandas>=2.0.0',
    'numpy>=1.24.0',
    'scipy>=1.11.0',
    'pyyaml>=6.0',
    'scikit-learn>=1.3.0',
    'xgboost>=2.0.0',
    'hmmlearn>=0.3.0',
    'shap>=0.44.0',
    'torch>=2.0.0',
    'transformers>=4.35.0',
    'stable-baselines3>=2.2.0',
    'gymnasium>=0.29.0',
    'cvxpy>=1.4.0',
    'PyPortfolioOpt>=1.5.5',
    'plotly>=5.17.0',
    'pyarrow>=13.0.0',
    'tqdm>=4.66.0',
]
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=False)

print('✅ Installation terminée!')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 2 : Clonage du repo GitHub
# ═══════════════════════════════════════════════════════════
import os

GITHUB_REPO = 'https://github.com/VOTRE_USERNAME/shariah-portfolio.git'

if not os.path.exists('shariah-portfolio'):
    os.system(f'git clone {GITHUB_REPO}')
    print('✅ Repo cloné')
else:
    os.system('cd shariah-portfolio && git pull')
    print('✅ Repo mis à jour')

os.chdir('shariah-portfolio')
sys.path.insert(0, 'src')
os.makedirs('data/cache', exist_ok=True)
print(f'📁 Répertoire : {os.getcwd()}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 3 : ⚙️ CONFIGURATION — MODIFIEZ ICI
# ═══════════════════════════════════════════════════════════
import yaml

# Charge la config de base
with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

# ── Paramètres modifiables ──────────────────────────────────

CAPITAL_INITIAL = 30_000          # € — votre capital
BROKER = 'fortuneo'               # 'fortuneo' | 'boursorama' | 'ibkr' | 'swissquote'
FILTRE_SHARIAH = 'intersection'   # 'intersection' | 'union' | 'msci_only' | 'djim_only'
METHODE_COV = 'ensemble'          # 'ensemble' | 'ledoit_wolf' | 'dcc_garch'
NB_TITRES_MIN = 15
NB_TITRES_MAX = 25
POIDS_MAX_TITRE = 0.10            # 10% max par titre
POIDS_MAX_SECTEUR = 0.30          # 30% max par secteur
DATE_DEBUT = '2004-01-01'         # 20 ans de backtest
DATE_FIN = '2024-12-31'

# Modules IA actifs
ACTIVER_HMM = True
ACTIVER_XGBOOST = True
ACTIVER_LSTM = True
ACTIVER_FINBERT = True           # plus lent (téléchargement modèle ~400MB)
ACTIVER_RL_AGENT = True

# Modèles extrêmes
MODELE_EXTREMES = 'ensemble'     # 'copula' | 'monte_carlo' | 'stress' | 'ensemble'

# ── Application à la config ────────────────────────────────
cfg['portfolio']['initial_capital'] = CAPITAL_INITIAL
cfg['broker']['name'] = BROKER
cfg['shariah']['filter_mode'] = FILTRE_SHARIAH
cfg['covariance']['method'] = METHODE_COV
cfg['portfolio']['min_stocks'] = NB_TITRES_MIN
cfg['portfolio']['max_stocks'] = NB_TITRES_MAX
cfg['portfolio']['max_weight_per_stock'] = POIDS_MAX_TITRE
cfg['portfolio']['max_weight_per_sector'] = POIDS_MAX_SECTEUR
cfg['data']['start_date'] = DATE_DEBUT
cfg['data']['end_date'] = DATE_FIN
cfg['ai']['hmm']['enabled'] = ACTIVER_HMM
cfg['ai']['xgboost']['enabled'] = ACTIVER_XGBOOST
cfg['ai']['lstm']['enabled'] = ACTIVER_LSTM
cfg['ai']['finbert']['enabled'] = ACTIVER_FINBERT
cfg['ai']['rl_agent']['enabled'] = ACTIVER_RL_AGENT
cfg['extreme_events']['method'] = MODELE_EXTREMES

print('✅ Configuration appliquée')
print(f'  Capital      : {CAPITAL_INITIAL:,}€')
print(f'  Broker       : {BROKER}')
print(f'  Shariah mode : {FILTRE_SHARIAH}')
print(f'  Covariance   : {METHODE_COV}')
print(f'  IA active    : HMM={ACTIVER_HMM} XGB={ACTIVER_XGBOOST} LSTM={ACTIVER_LSTM} FinBERT={ACTIVER_FINBERT} RL={ACTIVER_RL_AGENT}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 4 : Filtre Shariah — Construction de l'univers
# ═══════════════════════════════════════════════════════════
from universe import ShariahFilter

all_tickers = cfg['universe']['eu_tickers'] + cfg['universe']['us_tickers']
print(f'📊 Univers candidat : {len(all_tickers)} tickers')

sf = ShariahFilter(cfg)
universe_df = sf.build_universe(all_tickers, use_cache=True,
                                 cache_path=cfg['data']['cache_path'])
compliant_tickers = universe_df[universe_df['compliant']]['ticker'].tolist()

print(f'\n✅ Tickers Shariah conformes : {len(compliant_tickers)}/{len(all_tickers)}')
print('\n', universe_df[['ticker','msci','djim','compliant','sector','reason']].to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 5 : Pipeline de données (prix, dividendes, macro)
# ═══════════════════════════════════════════════════════════
from data_pipeline import DataPipeline

dp = DataPipeline(cfg)
data = dp.prepare(compliant_tickers, use_cache=True)

print(f'\n📈 Données chargées :')
print(f'  Prix    : {data["prices"].shape[1]} tickers × {data["prices"].shape[0]} jours')
print(f'  Macro   : {data["macro"].shape[1]} séries')
print(f'  Extrêmes: {data["extremes"]["is_extreme"].sum()} jours identifiés')
print(f'  NaN moyen : {data["returns"].isna().mean().mean():.1%}')
print('\n📅 Période :', data['prices'].index[0].date(), '→', data['prices'].index[-1].date())

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 6 : Entraînement des modèles IA
# ═══════════════════════════════════════════════════════════
from ml_engine import MLEngine
import os

engine = MLEngine(cfg)

# Essaie de charger les modèles depuis cache
models_cached = all([
    os.path.exists('data/cache/hmm.pkl'),
    os.path.exists('data/cache/xgb.pkl'),
    os.path.exists('data/cache/lstm.pkl'),
])

if models_cached:
    print('📂 Chargement modèles depuis cache...')
    engine.load('data/cache/')
else:
    print('🧠 Entraînement des modèles IA (10-20 min sur CPU, 3-5 min sur GPU)...')
    tickers_available = [t for t in compliant_tickers if t in data['returns'].columns]
    engine.fit(data['returns'], data['macro'], tickers_available)
    engine.save('data/cache/')

# Test : signaux sur la dernière date
tickers_ok = [t for t in compliant_tickers if t in data['returns'].columns]
signals = engine.generate_signals(data['returns'], data['macro'], tickers_ok)

print(f'\n🔮 Régime actuel    : {signals["regime"].upper()}')
print(f'   Probas [bull/bear/crisis] : {signals["regime_prob"].round(2)}')
print(f'   Volatilité prévue : {signals["vol_forecast"]:.1%}')
print(f'\n📊 Top 5 scores XGBoost :')
print(signals['scores'].sort_values(ascending=False).head())

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 7 : SHAP — Interprétabilité du modèle XGBoost
# ═══════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

tickers_ok = [t for t in compliant_tickers if t in data['returns'].columns]
shap_df = signals.get('shap_df')

if shap_df is not None and not shap_df.empty:
    mean_abs_shap = shap_df.abs().mean().sort_values(ascending=True).tail(12)
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#d62728' if v < 0 else '#2ca02c' for v in mean_abs_shap]
    mean_abs_shap.plot(kind='barh', ax=ax, color='#1f77b4')
    ax.set_title('🧠 SHAP Values — Importance des Features XGBoost\n(drivers des scores Alpha par ticker)', fontsize=13)
    ax.set_xlabel('|SHAP value| moyen')
    ax.axvline(0, color='black', linewidth=0.8)
    plt.tight_layout()
    plt.savefig('data/cache/shap_importance.png', bbox_inches='tight')
    plt.show()
    print('✅ SHAP importance plot généré')
else:
    print('ℹ️ SHAP non disponible (modèle non entraîné ou shap non installé)')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 8 : Entraînement Agent RL (PPO)
# ═══════════════════════════════════════════════════════════
from rl_agent import RLRebalancingAgent

rl_agent = RLRebalancingAgent(cfg)

if not rl_agent.load():
    print('🤖 Entraînement Agent PPO (rebalancing intelligent)...')
    tickers_ok = [t for t in compliant_tickers if t in data['returns'].columns]
    rl_agent.train(data['returns'], data['macro'], tickers_ok,
                   regime_series=engine.regime_series)

# Visualisation des décisions de l'agent
tickers_ok = [t for t in compliant_tickers if t in data['returns'].columns]
decisions = rl_agent.backtest_decisions(
    data['returns'].tail(252*3),
    data['macro'].tail(252*3),
    tickers_ok,
    regime_series=engine.regime_series
)

if not decisions.empty:
    print(f'\n🤖 Décisions Agent RL (3 dernières années) :')
    print(decisions['action'].value_counts())
    rebal_days = decisions[decisions['action_code'] > 0]
    print(f'\n  Rebalancings : {len(rebal_days)} en {len(decisions)} jours '
          f'({len(rebal_days)/len(decisions)*100:.1f}%)')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 9 : Black-Litterman + Optimisation
# ═══════════════════════════════════════════════════════════
from optimizer import BlackLittermanOptimizer

optimizer = BlackLittermanOptimizer(cfg)
tickers_ok = [t for t in compliant_tickers if t in data['returns'].columns]

print('🎯 Optimisation Black-Litterman...')
opt_result = optimizer.optimize(
    data['returns'],
    signals,
    tickers_ok,
    vol_forecast=signals.get('vol_forecast'),
)

w = opt_result['weights'].sort_values(ascending=False)
w_active = w[w > 0.001]

print(f'\n📊 Portefeuille optimal ({len(w_active)} titres actifs) :')
for ticker, weight in w_active.items():
    bar = '█' * int(weight * 100)
    print(f'  {ticker:<12} {weight:.1%}  {bar}')

# Allocation géographique
eu = sum(w[t] for t in w.index if '.' in t)
us = sum(w[t] for t in w.index if '.' not in t)
print(f'\n🌍 Allocation : Europe {eu:.1%} | US {us:.1%}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 10 : 🚀 BACKTEST WALK-FORWARD COMPLET (20 ans)
# ═══════════════════════════════════════════════════════════
from backtest import WalkForwardBacktest
from costs import RebalancingEngine

print('🚀 Lancement du backtest walk-forward 20 ans...')
print('   (peut prendre 5-15 min selon les données disponibles)')

bt = WalkForwardBacktest(cfg)
reb_engine = RebalancingEngine(cfg)

tickers_ok = [t for t in compliant_tickers if t in data['returns'].columns]
results = bt.run(
    data, tickers_ok,
    ml_engine=engine,
    optimizer=optimizer,
    rebalance_engine=reb_engine,
    rl_agent=rl_agent,
)

m = results['metrics']
print('\n' + '═'*55)
print('          RÉSULTATS BACKTEST 20 ANS')
print('═'*55)
print(f"  Rendement total      : {m.get('total_return', 0):.1%}")
print(f"  Rendement annualisé  : {m.get('annualized_return', 0):.1%}")
print(f"  Volatilité annuelle  : {m.get('volatility', 0):.1%}")
print(f"  Sharpe ratio         : {m.get('sharpe_ratio', 0):.2f}")
print(f"  Sortino ratio        : {m.get('sortino_ratio', 0):.2f}")
print(f"  Max Drawdown         : {m.get('max_drawdown', 0):.1%}")
print(f"  Calmar ratio         : {m.get('calmar_ratio', 0):.2f}")
print(f"  VaR 95%              : {m.get('var_95', 0):.2%}")
print(f"  CVaR 95%             : {m.get('cvar_95', 0):.2%}")
print('═'*55)

# Benchmarks
for name, bm in results.get('bench_metrics', {}).items():
    print(f"  [{name}] Sharpe: {bm.get('sharpe_ratio',0):.2f} | "
          f"Ann.Ret: {bm.get('annualized_return',0):.1%} | "
          f"Max DD: {bm.get('max_drawdown',0):.1%}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 11 : 📊 VISUALISATIONS COMPLÈTES
# ═══════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
import numpy as np

plt.style.use('seaborn-v0_8-darkgrid')
fig = plt.figure(figsize=(18, 22))
gs = gridspec.GridSpec(4, 2, figure=fig, hspace=0.4, wspace=0.35)

port_ret = results['portfolio_returns'].dropna()
cum_port = (1 + port_ret).cumprod()

# ── 1. Valeur du Portefeuille ────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
capital = cfg['portfolio']['initial_capital']
(cum_port * capital).plot(ax=ax1, label='🕌 Shariah Portfolio', color='#2ecc71', linewidth=2)

colors_bench = ['#e74c3c', '#3498db', '#9b59b6']
for (name, bret), color in zip(results.get('benchmarks', {}).items(), colors_bench):
    bret_aligned = bret.reindex(port_ret.index).fillna(0)
    (((1 + bret_aligned).cumprod()) * capital).plot(
        ax=ax1, label=name, color=color, linewidth=1.5, alpha=0.7, linestyle='--')

ax1.set_title('📈 Valeur du Portefeuille Shariah vs Benchmarks', fontsize=14, fontweight='bold')
ax1.set_ylabel('Valeur (€)')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}€'))
ax1.legend(fontsize=10)
ax1.fill_between(cum_port.index, capital, cum_port * capital,
                  where=cum_port > 1, alpha=0.1, color='green')
ax1.fill_between(cum_port.index, capital, cum_port * capital,
                  where=cum_port < 1, alpha=0.1, color='red')

# ── 2. Drawdown ─────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
roll_max = cum_port.cummax()
dd = (cum_port - roll_max) / roll_max
dd.plot(ax=ax2, color='#e74c3c', linewidth=1)
ax2.fill_between(dd.index, dd, 0, alpha=0.3, color='#e74c3c')
ax2.set_title('📉 Drawdown', fontsize=12)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax2.axhline(results['metrics'].get('max_drawdown', 0), color='darkred',
             linestyle=':', label=f"Max DD: {results['metrics'].get('max_drawdown', 0):.1%}")
ax2.legend(fontsize=9)

# ── 3. Distribution des Rendements ─────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
ax3.hist(port_ret * 100, bins=80, color='#3498db', alpha=0.7, edgecolor='none')
var95 = results['metrics'].get('var_95', 0) * 100
cvar95 = results['metrics'].get('cvar_95', 0) * 100
ax3.axvline(var95, color='orange', linewidth=2, label=f'VaR 95%: {var95:.2f}%')
ax3.axvline(cvar95, color='red', linewidth=2, label=f'CVaR 95%: {cvar95:.2f}%')
ax3.set_title('📊 Distribution des Rendements Journaliers', fontsize=12)
ax3.set_xlabel('Rendement (%)')
ax3.legend(fontsize=9)

# ── 4. Allocation Actuelle ───────────────────────────────────
ax4 = fig.add_subplot(gs[2, 0])
w_active = opt_result['weights'][opt_result['weights'] > 0.005].sort_values(ascending=True)
colors_alloc = ['#27ae60' if '.' in t else '#2980b9' for t in w_active.index]
bars = ax4.barh(range(len(w_active)), w_active.values * 100, color=colors_alloc)
ax4.set_yticks(range(len(w_active)))
ax4.set_yticklabels(w_active.index, fontsize=9)
ax4.set_title('🏦 Allocation Portefeuille (vert=EU, bleu=US)', fontsize=12)
ax4.set_xlabel('Poids (%)')
for i, (bar, val) in enumerate(zip(bars, w_active.values)):
    ax4.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
             f'{val:.1%}', va='center', fontsize=8)

# ── 5. Régimes de Marché ─────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 1])
regime_hist = results.get('regime_history', pd.Series())
if not regime_hist.empty:
    regime_colors = {'bull': '#27ae60', 'bear': '#e74c3c', 'crisis': '#8e44ad', 'unknown': '#95a5a6'}
    for date, regime in regime_hist.items():
        ax5.axvspan(date, date + pd.DateOffset(months=12),
                    alpha=0.3, color=regime_colors.get(regime, '#95a5a6'))
    (cum_port * capital).plot(ax=ax5, color='black', linewidth=1.5)
    from matplotlib.patches import Patch
    legend_els = [Patch(facecolor=c, alpha=0.5, label=r)
                  for r, c in regime_colors.items() if r != 'unknown']
    ax5.legend(handles=legend_els, fontsize=9)
ax5.set_title('🔮 Régimes HMM (bull/bear/crisis)', fontsize=12)
ax5.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}€'))

# ── 6. Coûts de Transaction ──────────────────────────────────
ax6 = fig.add_subplot(gs[3, :])
trade_hist = results.get('trade_history', pd.DataFrame())
if not trade_hist.empty and 'date' in trade_hist.columns:
    trade_hist['date'] = pd.to_datetime(trade_hist['date'])
    fees_by_month = trade_hist.groupby(
        trade_hist['date'].dt.to_period('Q'))['fee'].sum()
    fees_by_month.index = fees_by_month.index.to_timestamp()
    ax6.bar(fees_by_month.index, fees_by_month.values,
             width=60, color='#e67e22', alpha=0.8, label='Frais broker')
    ax6.set_title(f'💰 Frais de Transaction Trimestriels ({BROKER.title()})', fontsize=12)
    ax6.set_ylabel('Frais (€)')
    total_fees = trade_hist['fee'].sum()
    ax6.text(0.02, 0.95, f'Total frais : {total_fees:,.0f}€', transform=ax6.transAxes,
              fontsize=10, verticalalignment='top',
              bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
else:
    ax6.text(0.5, 0.5, 'Historique des trades non disponible',
              ha='center', va='center', transform=ax6.transAxes)
ax6.legend()

plt.suptitle('🕌 Shariah Efficient Portfolio — Analyse Complète 20 ans\n'
              f'(HMM + XGBoost + LSTM + FinBERT + PPO | {BROKER.title()} | CTO France)',
              fontsize=14, fontweight='bold', y=0.98)

plt.savefig('data/cache/backtest_results.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Dashboard sauvegardé : data/cache/backtest_results.png')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 12 : 📋 RAPPORT FINAL — Comparaison des coûts
# ═══════════════════════════════════════════════════════════
from costs import BrokerCosts

print('═'*60)
print('    ANALYSE COMPARATIVE DES BROKERS')
print('═'*60)

capital_test = cfg['portfolio']['initial_capital']
n_us = len([t for t in compliant_tickers if '.' not in t and t in data['returns'].columns])
n_eu = len([t for t in compliant_tickers if '.' in t and t in data['returns'].columns])

for broker_name in ['fortuneo', 'boursorama', 'ibkr', 'swissquote']:
    cfg_copy = dict(cfg)
    cfg_copy['broker'] = dict(cfg['broker'])
    cfg_copy['broker']['name'] = broker_name
    bc = BrokerCosts(cfg_copy)

    # Coût construction portefeuille initial
    amount_eu = capital_test * 0.5 / max(n_eu, 1)
    amount_us = capital_test * 0.5 / max(n_us, 1)
    init_cost = n_eu * bc.transaction_cost(amount_eu, 'EU') + \
                n_us * bc.transaction_cost(amount_us, 'US')

    # Coût rebalancing annuel (hypothèse : 30% du portif rebalancé)
    rebal_cost = (n_eu * 0.3 * bc.transaction_cost(amount_eu, 'EU') +
                  n_us * 0.3 * bc.transaction_cost(amount_us, 'US'))

    custody = bc.annual_custody_fee(capital_test)
    init_pct = init_cost / capital_test * 100
    rebal_pct = rebal_cost / capital_test * 100

    print(f'\n  {broker_name.upper():<15}')
    print(f'    Init cost      : {init_cost:>7.0f}€  ({init_pct:.2f}%)')
    print(f'    Rebal/an       : {rebal_cost:>7.0f}€  ({rebal_pct:.2f}%)')
    print(f'    Garde/an       : {custody:>7.0f}€')
    print(f'    Drag total/an  : {(rebal_pct + custody/capital_test*100):.2f}%')

print('\n═'*60)
print('💡 Note : avec 30,000€ + Fortuneo, chaque ordre US coûte')
print('   50€ fixe = 1.67% de drag pour une position de 3,000€')
print('   → Boursorama ou IBKR fortement recommandés pour US')
print('═'*60)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 13 : 💾 Export des résultats
# ═══════════════════════════════════════════════════════════
import json

# Export CSV des rendements
results['portfolio_returns'].to_csv('data/cache/portfolio_returns.csv')

# Export métriques JSON
metrics_export = {k: (float(v) if isinstance(v, (np.floating, np.integer)) else v)
                  for k, v in results['metrics'].items()}
with open('data/cache/metrics.json', 'w') as f:
    json.dump(metrics_export, f, indent=2)

# Export poids optimaux
opt_result['weights'].to_csv('data/cache/optimal_weights.csv')

# Téléchargement depuis Colab
try:
    from google.colab import files
    print('📥 Téléchargement des fichiers résultats...')
    files.download('data/cache/backtest_results.png')
    files.download('data/cache/metrics.json')
    files.download('data/cache/optimal_weights.csv')
except ImportError:
    print('ℹ️ Fichiers disponibles dans data/cache/')

print('\n✅ Export terminé !')
print('  📊 data/cache/backtest_results.png')
print('  📋 data/cache/metrics.json')
print('  💼 data/cache/optimal_weights.csv')
print('  📈 data/cache/portfolio_returns.csv')